# CE49X Lab 3: Where Should You Open a Gas Station in Istanbul?
## A Traffic-Based Site Selection Analysis

**Instructor:** Dr. Eyuphan Koc  
**Department of Civil Engineering, Bogazici University**  
**Semester:** Spring 2026

---

## Background

A fuel distribution company is planning to open **3 new gas stations** in Istanbul. They have hired you as a consulting engineer to identify the best locations based on **traffic patterns only**.

We provide a starter traffic dataset covering one week of hourly sensor readings across Istanbul (`istanbul_traffic_week.csv` + `sensor_coords.csv`). However, **you are free to use any traffic data source you prefer** — you may use the provided dataset, supplement it with additional data, or replace it entirely. Some options:

- **Provided dataset:** `istanbul_traffic_week.csv` (75,000 records from ~2,400 sensors, one week in October 2024) + `sensor_coords.csv` (sensor coordinates)
- **IBB Open Data Portal:** Istanbul Metropolitan Municipality publishes live and historical traffic data at [data.ibb.gov.tr](https://data.ibb.gov.tr). You can query their APIs for broader coverage or more recent data.
- **Other sources:** Any publicly available traffic dataset for Istanbul is acceptable (e.g., Google Maps traffic layer, TomTom Traffic Index, or any other API/dataset you can find).

**Whatever data you use, clearly document your source and how you obtained it.**

Your job is to:
1. **Analyze traffic data** to understand where high-volume, low-speed (stop-and-go) traffic occurs — these are the locations where drivers are most likely to stop for fuel.
2. **Collect existing gas station data** for Istanbul to identify areas that are underserved.
3. **Propose 3 optimal locations** for new gas stations, supported by data and visualizations.

## Provided Data (Optional Starting Point)

The following files are included in the course repository. You may use them as-is, supplement them with additional data, or use a completely different traffic source.

### `istanbul_traffic_week.csv`

| Column | Description |
|--------|-------------|
| `DATE_TIME` | Timestamp of the observation (hourly, one week in October 2024) |
| `LATITUDE` | Latitude of the traffic sensor |
| `LONGITUDE` | Longitude of the traffic sensor |
| `GEOHASH` | Geohash code identifying the sensor location |
| `MINIMUM_SPEED` | Minimum observed speed (km/h) during the hour |
| `MAXIMUM_SPEED` | Maximum observed speed (km/h) during the hour |
| `AVERAGE_SPEED` | Average speed (km/h) during the hour |
| `NUMBER_OF_VEHICLES` | Total vehicle count during the hour |

### `sensor_coords.csv`

| Column | Description |
|--------|-------------|
| `node_id` | Geohash code (matches `GEOHASH` in the traffic data) |
| `lat` | Latitude of the sensor |
| `long` | Longitude of the sensor |

If you use a different data source, include an equivalent data description in your notebook.

## Deliverables

Your notebook must include the following:

### 1. Traffic Data — Source & Exploration
- **Document your traffic data source.** If you use the provided dataset, state that. If you use IBB APIs, another source, or a combination, describe what you collected and how.
- Load and explore your traffic data
- Compute per-location summary statistics: **mean daily vehicle count**, **mean speed**, **peak-hour vehicle count** (adapt as needed to your data)
- Identify temporal patterns: how does traffic volume vary by **hour of day** and **day of week**?
- Identify the **top 20 highest-traffic locations** by total vehicle count

### 2. Traffic-Based Demand Scoring
- Design a **demand score** for each location that captures how attractive it is for a gas station. Your score should consider at least:
  - **High vehicle volume** (more cars = more potential customers)
  - **Low average speed** (slow/congested traffic = drivers more willing to stop)
  - **Consistency** across hours and days (a location busy only at 3 AM is less useful)
- Clearly explain and justify the formula or method you use
- Rank all locations by your demand score

### 3. Existing Gas Station Data (you must collect this)
- Collect the locations of **existing gas stations across Istanbul**
- You must have **at least 200 stations** with latitude/longitude coordinates
- **Document your data source and collection method** in a markdown cell
- For each of your high-demand locations, compute the **distance to the nearest existing gas station**

### 4. Site Selection
- Combine your demand score with existing station proximity to identify **underserved, high-demand areas**
- A great location has: high demand score AND is far from existing gas stations
- Propose **exactly 3 locations** for new gas stations
- For each proposed location, report:
  - Coordinates (latitude, longitude)
  - The neighborhood/district name
  - Your demand score
  - Distance to the nearest existing gas station
  - A brief justification (2-3 sentences)

### 5. Visualizations
- Create **at least three plots/maps**. Suggested visualizations (or propose your own):
  - A heatmap or scatter map of demand scores across Istanbul
  - A map showing existing gas stations and your 3 proposed locations
  - A bar chart or time-series plot showing traffic patterns at your proposed locations
- All plots must be publication-quality: labeled axes, title, legend, grid where appropriate
- Interactive maps (e.g., folium) are encouraged but not required

### 6. Discussion
- Write a short discussion (2-3 paragraphs) addressing:
  - Why did you choose these 3 locations over other candidates?
  - What **limitations** does a traffic-only analysis have? What other factors would a real site selection study consider (e.g., land cost, zoning, competition, road type)?
  - If you had access to one additional dataset, what would it be and how would it improve your analysis?

## Hints

- **Haversine formula** for distance between two GPS coordinates:

$$d = 2R \arcsin\left(\sqrt{\sin^2\left(\frac{\Delta\phi}{2}\right) + \cos(\phi_1)\cos(\phi_2)\sin^2\left(\frac{\Delta\lambda}{2}\right)}\right)$$

  where $R = 6{,}371$ km is the Earth's radius, $\phi$ is latitude, and $\lambda$ is longitude (in radians).

- **Normalizing scores:** When combining metrics with different scales (e.g., vehicle count vs. speed), normalize each to a 0-1 range first:

$$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

- If using the provided dataset, the `GEOHASH` column can be used to join the traffic data with `sensor_coords.csv` via the `node_id` column.

- Think about whether **weekday** vs. **weekend** traffic patterns matter for a gas station business.

## Grading

| Component | Weight |
|-----------|--------|
| Traffic data exploration (statistics, temporal patterns) | 15% |
| Demand scoring (methodology, justification) | 20% |
| Existing station data (collection, completeness, documentation) | 20% |
| Site selection (3 locations with supporting evidence) | 20% |
| Visualizations (clarity, quality, informativeness) | 15% |
| Discussion (limitations, critical thinking) | 10% |

## Submission

1. Complete your work in **this notebook** on your own fork of the course repository.
2. Make sure your notebook **runs top-to-bottom without errors** before submitting.
3. Commit and push your completed notebook to your fork.
4. We will grade directly from your fork — there is no separate upload. Make sure your latest work is pushed before the deadline.

---
## Your Work Starts Here

TRAFFIC DATA: SOURCE &&&&& EXPLORATION 

In [13]:
# ============================================================
# SECTION 1 — Traffic Data Exploration
# Istanbul Gas Station Site Selection Analysis
# Dataset: istanbul_traffic_week.csv (October 2024)
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Change this to the path of your CSV file ────────────────
FILE_PATH = "istanbul_traffic_week.csv"
# ────────────────────────────────────────────────────────────


# ============================================================
# STEP 1 — Load the dataset
# ============================================================
df = pd.read_csv("D:\CE49X\CE49X\Week03_NumPy_Pandas\istanbul_traffic_week.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)


# ============================================================
# STEP 2 — Convert DATE_TIME to datetime format
# ============================================================
df["DATE_TIME"] = pd.to_datetime(df["DATE_TIME"])

print("\nDATE_TIME dtype after conversion:", df["DATE_TIME"].dtype)
print("Date range:", df["DATE_TIME"].min(), "→", df["DATE_TIME"].max())


# ============================================================
# STEP 3 — Extract temporal features
# ============================================================
df["hour"]        = df["DATE_TIME"].dt.hour          # 0–23
df["day_of_week"] = df["DATE_TIME"].dt.dayofweek     # 0=Monday, 6=Sunday
df["day_name"]    = df["DATE_TIME"].dt.day_name()    # "Monday", "Tuesday" …
df["date"]        = df["DATE_TIME"].dt.date          # calendar date only

print("\nNew temporal columns added:")
print(df[["DATE_TIME", "hour", "day_of_week", "day_name", "date"]].head(8))


# ============================================================
# STEP 4 — Per-location summary statistics
# Each unique location is identified by its GEOHASH cell.
# ============================================================

# ── 4a. Mean daily vehicle count ────────────────────────────
# First sum all vehicles per location per day,
# then average those daily totals across the 7-day window.
daily_totals = (
    df.groupby(["GEOHASH", "date"])["NUMBER_OF_VEHICLES"]
    .sum()
    .reset_index()
    .rename(columns={"NUMBER_OF_VEHICLES": "daily_vehicles"})
)
mean_daily = (
    daily_totals.groupby("GEOHASH")["daily_vehicles"]
    .mean()
    .rename("mean_daily_vehicles")
)

# ── 4b. Mean average speed ───────────────────────────────────
mean_speed = (
    df.groupby("GEOHASH")["AVERAGE_SPEED"]
    .mean()
    .rename("mean_avg_speed")
)

# ── 4c. Peak-hour vehicle count ─────────────────────────────
# For every location, find the hour-of-day that has the highest
# average vehicle count across all 7 days.
hourly_avg = (
    df.groupby(["GEOHASH", "hour"])["NUMBER_OF_VEHICLES"]
    .mean()
    .reset_index()
)
peak_idx   = hourly_avg.groupby("GEOHASH")["NUMBER_OF_VEHICLES"].idxmax()
peak_hour_vehicles = (
    hourly_avg.loc[peak_idx]
    .set_index("GEOHASH")["NUMBER_OF_VEHICLES"]
    .rename("peak_hour_vehicles")
)


# ============================================================
# STEP 5 — Traffic consistency
# Consistency = 1 / std(hourly vehicle counts)
# High value → low variance → demand is stable and predictable
# ============================================================
hourly_std = (
    df.groupby("GEOHASH")["NUMBER_OF_VEHICLES"]
    .std()
    .rename("hourly_std")
)
# Add a tiny epsilon to avoid division by zero for single-row locations
traffic_consistency = (1.0 / (hourly_std + 1e-6)).rename("traffic_consistency")

# Fill NaN values (locations with only one observation have std = NaN)
traffic_consistency = traffic_consistency.fillna(traffic_consistency.max())


# ============================================================
# Total weekly vehicle count — used for Top-20 ranking
# ============================================================
total_weekly = (
    df.groupby("GEOHASH")["NUMBER_OF_VEHICLES"]
    .sum()
    .rename("total_weekly_vehicles")
)

# ── Representative lat/lon for each location ─────────────────
coords = df.groupby("GEOHASH")[["LATITUDE", "LONGITUDE"]].first()


# ============================================================
# STEP 8 — Assemble the summary dataframe
# ============================================================
summary = pd.concat(
    [coords, mean_daily, mean_speed,
     peak_hour_vehicles, traffic_consistency, total_weekly],
    axis=1
).reset_index()

summary.columns = [
    "GEOHASH", "LATITUDE", "LONGITUDE",
    "mean_daily_vehicles", "mean_avg_speed",
    "peak_hour_vehicles",  "traffic_consistency",
    "total_weekly_vehicles"
]

print("\n── Summary dataframe ──────────────────────────────────")
print(f"  Locations: {len(summary):,}")
print(summary.head(10).to_string(index=False))
print("\nDescriptive statistics:")
print(summary.describe().round(2))


# ============================================================
# STEP 6 — Temporal pattern plots
# ============================================================

DAY_ORDER = ["Monday", "Tuesday", "Wednesday",
             "Thursday", "Friday", "Saturday", "Sunday"]

# ── Plot A: Average traffic by hour of day (line plot) ───────
hourly_city = df.groupby("hour")["NUMBER_OF_VEHICLES"].mean()

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    hourly_city.index,
    hourly_city.values,
    color="#E84545",
    linewidth=2.5,
    marker="o",
    markersize=5,
    label="Avg vehicles per sensor per hour"
)
ax.fill_between(hourly_city.index, hourly_city.values,
                color="#E84545", alpha=0.12)

ax.set_title("Average Traffic Volume by Hour of Day\n"
             "Istanbul — October 2024",
             fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Hour of Day (0 = midnight,  12 = noon,  23 = 11 pm)",
              fontsize=11)
ax.set_ylabel("Avg Number of Vehicles", fontsize=11)
ax.set_xticks(range(0, 24))
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig("plot_hourly_traffic.png", dpi=150)
plt.show()
print("Saved → plot_hourly_traffic.png")


# ── Plot B: Average traffic by day of week (bar plot) ────────
dow_avg = (
    df.groupby("day_name")["NUMBER_OF_VEHICLES"]
    .mean()
    .reindex(DAY_ORDER)
)

# Build a colour gradient from light yellow to dark red
bar_colors = [
    "#FFF176", "#FFD54F", "#FFB300",
    "#FB8C00", "#F4511E", "#E53935", "#B71C1C"
]

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(
    dow_avg.index,
    dow_avg.values,
    color=bar_colors,
    edgecolor="white",
    linewidth=0.8
)
# Add value labels on top of each bar
ax.bar_label(bars, fmt="%.0f", padding=4, fontsize=9)

ax.set_title("Average Traffic Volume by Day of Week\n"
             "Istanbul — October 2024",
             fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Day of Week", fontsize=11)
ax.set_ylabel("Avg Number of Vehicles", fontsize=11)
ax.set_ylim(0, dow_avg.max() * 1.15)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("plot_dayofweek_traffic.png", dpi=150)
plt.show()
print("Saved → plot_dayofweek_traffic.png")


# ============================================================
# STEP 7 — Top 20 highest-traffic locations
# ============================================================
top20 = (
    summary
    .nlargest(20, "total_weekly_vehicles")
    [["GEOHASH", "LATITUDE", "LONGITUDE",
      "mean_daily_vehicles", "mean_avg_speed",
      "total_weekly_vehicles"]]
    .reset_index(drop=True)
)
top20.index += 1   # start ranking at 1 instead of 0

print("\n── Top 20 Highest-Traffic Locations ───────────────────")
print(top20.to_string())

# Bar chart of the Top 20
fig, ax = plt.subplots(figsize=(12, 6))

ax.barh(
    range(20),
    top20["total_weekly_vehicles"].values[::-1],
    color="#1565C0",
    alpha=0.8,
    edgecolor="white"
)
ax.set_yticks(range(20))
ax.set_yticklabels(top20["GEOHASH"].values[::-1], fontsize=8)
ax.set_title("Top 20 Locations by Total Weekly Vehicle Count\n"
             "Istanbul — October 2024",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Total Weekly Vehicles", fontsize=11)
ax.set_ylabel("Geohash (location ID)", fontsize=11)
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"{int(x):,}"
))

plt.tight_layout()
plt.savefig("plot_top20_locations.png", dpi=150)
plt.show()
print("Saved → plot_top20_locations.png")


# ============================================================
# FINAL — Show the clean summary dataframe
# ============================================================
print("\n── Final Summary Dataframe (first 20 rows) ────────────")
print(summary.head(20).round(2).to_string(index=False))

print("\n✅  Section 1 complete.")
print(f"   {len(summary):,} sensor locations processed.")
print( "   Outputs: plot_hourly_traffic.png | "
       "plot_dayofweek_traffic.png | plot_top20_locations.png")

Shape: (75000, 8)

Columns: ['DATE_TIME', 'LATITUDE', 'LONGITUDE', 'GEOHASH', 'MINIMUM_SPEED', 'MAXIMUM_SPEED', 'AVERAGE_SPEED', 'NUMBER_OF_VEHICLES']

First 5 rows:
             DATE_TIME   LATITUDE  LONGITUDE GEOHASH  MINIMUM_SPEED  \
0  2024-10-01 00:00:00  41.119080  29.042358  sxk9uv             49   
1  2024-10-01 00:00:00  41.064148  29.064331  sxk9t7              8   
2  2024-10-01 00:00:00  41.091614  29.031372  sxk9u8             10   
3  2024-10-01 00:00:00  41.108093  29.086304  sxk9vg              2   
4  2024-10-01 00:00:00  41.113586  29.042358  sxk9uu              7   

   MAXIMUM_SPEED  AVERAGE_SPEED  NUMBER_OF_VEHICLES  
0             67             59                   3  
1             48             27                   6  
2            149             77                 180  
3             60             39                  12  
4             88             46                  16  

Data types:
DATE_TIME              object
LATITUDE              float64
LONGITUDE 

SECTION 2

In [21]:
# ============================================================
# SECTION 2 — Demand Score Model
# Istanbul Gas Station Site Selection
# ============================================================
# Assumes Section 1 has already been run.
# The dataframe `summary` must exist in the same session.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


# ============================================================
# STEP 1 — Feature Engineering & Min-Max Normalisation
# Four input signals are derived and scaled to [0, 1].
# Each transformation addresses a specific modelling weakness
# identified in the original linear scoring approach.
# ============================================================

def minmax(series):
    """Min-Max scale a Series to [0, 1]."""
    col_min = series.min()
    col_max = series.max()
    return (series - col_min) / (col_max - col_min)


# ── Signal 1: Log-Scaled Daily Volume ────────────────────────
# Raw vehicle counts are heavily right-skewed: a small number
# of bridge and highway corridors record counts 10–20× the
# city median.  Applying log1p compresses these extremes while
# preserving rank order, preventing a single dominant corridor
# from overwhelming the composite score.
summary["log_mean_daily_vehicles"]      = np.log1p(summary["mean_daily_vehicles"])
summary["norm_log_mean_daily_vehicles"] = minmax(summary["log_mean_daily_vehicles"])

# ── Signal 2: Peak-Hour Ratio (Relative Intensity) ───────────
# Absolute peak-hour counts are strongly correlated with daily
# volume — using both would double-count traffic size.  The
# ratio peak / daily isolates rush-hour intensity as a share
# of baseline demand, capturing how pronounced the peak is
# independently of the corridor's overall throughput.
summary["peak_hour_ratio"]      = (
    summary["peak_hour_vehicles"] / summary["mean_daily_vehicles"]
)
summary["norm_peak_hour_ratio"] = minmax(summary["peak_hour_ratio"])

# ── Signal 3: Traffic Consistency (unchanged) ────────────────
# Inverse of hourly standard deviation.  A high score means
# demand is stable and predictable across the week — a
# desirable property for retail fuel investment.
summary["norm_traffic_consistency"] = minmax(summary["traffic_consistency"])

# ── Signal 4: Congestion Score (Inverted Speed) ───────────────
# High average speed indicates a free-flow highway or bridge
# where drivers are unlikely to stop.  Inverting normalised
# speed (1 − norm) produces a congestion proxy: moderate
# congestion raises the score, reflecting higher stop
# likelihood and walk-in potential near the site.
summary["norm_mean_avg_speed"] = minmax(summary["mean_avg_speed"])
summary["congestion_score"]    = 1 - summary["norm_mean_avg_speed"]

print("── Engineered features added ──────────────────────────")
eng_cols = ["GEOHASH",
            "norm_log_mean_daily_vehicles",
            "norm_peak_hour_ratio",
            "norm_traffic_consistency",
            "congestion_score"]
print(summary[eng_cols].head(5).round(4).to_string(index=False))


# ============================================================
# STEP 2 — Composite Demand Score
# Weights reflect the relative importance of each signal:
#   0.45 — log daily volume      → overall traffic scale
#                                   (log-scaled to reduce bridge bias)
#   0.20 — peak-hour ratio       → rush-hour intensity relative
#                                   to baseline (avoids double-count)
#   0.20 — traffic consistency   → reliability / predictability
#   0.15 — congestion score      → inverted speed; higher score
#                                   where drivers are more likely
#                                   to slow down and stop
# Weights sum to 1.0.
# ============================================================

summary["DemandScore"] = (
    0.45 * summary["norm_log_mean_daily_vehicles"] +
    0.20 * summary["norm_peak_hour_ratio"]         +
    0.20 * summary["norm_traffic_consistency"]     +
    0.15 * summary["congestion_score"]
)

print("\n── DemandScore column added ───────────────────────────")
print(summary[["GEOHASH", "log_mean_daily_vehicles",
               "peak_hour_ratio", "DemandScore"]].head(5).round(4).to_string(index=False))


# ============================================================
# STEP 3 — Rank all locations by DemandScore (descending)
# Rank 1 = highest demand
# ============================================================

summary["DemandRank"] = (
    summary["DemandScore"]
    .rank(ascending=False, method="min")
    .astype(int)
)

print("\n── Rankings assigned ──────────────────────────────────")
print(f"  Total locations ranked: {len(summary):,}")


# ============================================================
# STEP 4 — Extract the Top 15 highest-demand locations
# ============================================================

top15 = (
    summary
    .nsmallest(15, "DemandRank")          # lowest rank number = highest score
    [["DemandRank", "GEOHASH", "LATITUDE", "LONGITUDE",
      "mean_daily_vehicles", "peak_hour_vehicles",
      "traffic_consistency", "mean_avg_speed", "DemandScore"]]
    .reset_index(drop=True)
)
top15.index += 1                           # display index starts at 1


# ============================================================
# STEP 5 — Print Top 15 table and DemandScore statistics
# ============================================================

print("\n── Top 15 Locations by Demand Score ───────────────────")
print(top15.round(4).to_string())

print("\n── DemandScore — Descriptive Statistics ───────────────")
print(summary["DemandScore"].describe().round(4))


# ============================================================
# STEP 6 — Bar plot: Top 15 locations by DemandScore
# ============================================================

# Shorten geohash labels for readability on the chart
top15["label"] = (
    top15["GEOHASH"] + "\n" +
    top15["LATITUDE"].round(3).astype(str) + "°N, " +
    top15["LONGITUDE"].round(3).astype(str) + "°E"
)

# Colour gradient from light to dark blue (rank 15 → rank 1)
cmap   = plt.cm.Blues
colors = [cmap(0.4 + 0.6 * i / 14) for i in range(15)]

fig, ax = plt.subplots(figsize=(12, 8))

bars = ax.barh(
    range(15),
    top15["DemandScore"].values[::-1],     # highest score at the top
    color=colors[::-1],
    edgecolor="white",
    linewidth=0.7,
    height=0.65,
)

# Add score labels at the end of each bar
for bar, score in zip(bars, top15["DemandScore"].values[::-1]):
    ax.text(
        bar.get_width() + 0.003,
        bar.get_y() + bar.get_height() / 2,
        f"{score:.4f}",
        va="center", ha="left",
        fontsize=8.5, color="#333333"
    )

# Y-axis labels (geohash + coordinates)
ax.set_yticks(range(15))
ax.set_yticklabels(top15["label"].values[::-1], fontsize=8.5)

ax.set_title(
    "Top 15 Locations by Demand Score\n"
    "Istanbul Gas Station Site Selection — October 2024",
    fontsize=14, fontweight="bold", pad=14
)
ax.set_xlabel("Demand Score  (weighted composite, 0 – 1)", fontsize=11)
ax.set_ylabel("Location (Geohash  |  Coordinates)", fontsize=11)
ax.set_xlim(0, top15["DemandScore"].max() * 1.18)
ax.grid(axis="x", linestyle="--", alpha=0.45)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))

# Annotation box — weight legend
weight_text = (
    "Weights\n"
    "Log daily volume    : 0.45\n"
    "Peak-hour ratio     : 0.20\n"
    "Traffic consistency : 0.20\n"
    "Congestion score    : 0.15"
)
ax.text(
    0.98, 0.02, weight_text,
    transform=ax.transAxes,
    fontsize=8, va="bottom", ha="right",
    bbox=dict(boxstyle="round,pad=0.5", fc="white",
              ec="#BBBBBB", alpha=0.85)
)

plt.tight_layout()
plt.savefig("plot_top15_demand_score.png", dpi=150)
plt.show()
print("\nSaved → plot_top15_demand_score.png")


# ============================================================
# DONE
# ============================================================
print("\nSection 2 complete — Demand model successfully applied.")

── Engineered features added ──────────────────────────
GEOHASH  norm_log_mean_daily_vehicles  norm_peak_hour_ratio  norm_traffic_consistency  congestion_score
 sx7chk                        0.4063                0.2981                       0.0            0.2834
 sx7chm                        0.4940                0.1934                       0.0            0.4244
 sx7cht                        0.4784                0.2362                       0.0            0.7584
 sx7chw                        0.4424                0.3172                       0.0            0.4647
 sx7chx                        0.4706                0.2593                       0.0            0.4322

── DemandScore column added ───────────────────────────
GEOHASH  log_mean_daily_vehicles  peak_hour_ratio  DemandScore
 sx7chk                   4.2047           0.6364       0.2850
 sx7chm                   4.9628           0.4331       0.3247
 sx7cht                   4.8283           0.5161       0.3763
 sx7chw    

SECTION 3

In [22]:
# ============================================================
# SECTION 3 — Existing Gas Station Competition Analysis
# Istanbul Gas Station Site Selection
# ============================================================
# Prerequisites:
#   - Section 1 must have run  →  `summary` dataframe exists
#   - Section 2 must have run  →  `top15` dataframe with
#                                  DemandScore / DemandRank exists
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches


# ============================================================
# STEP 1 — Load the existing gas stations dataset
# ============================================================

# ── Change this path if your CSV is in a different folder ───
STATIONS_FILE = "D:\CE49X\CE49X\Week03_NumPy_Pandas\existing_gas_stations_istanbul.csv.csv"
# ────────────────────────────────────────────────────────────

stations = pd.read_csv(STATIONS_FILE)

# Rename columns to clean, consistent names
stations = stations.rename(columns={
    "name": "station_name",
    "@lat": "latitude",
    "@lon": "longitude"
})

# Fill missing station names with a readable placeholder
stations["station_name"] = stations["station_name"].fillna("Unnamed Station")


# ============================================================
# STEP 2 — Inspect the stations dataframe
# ============================================================

print("=" * 60)
print("SECTION 3 — Competition & Distance Analysis")
print("=" * 60)

print(f"\n── Gas Stations Loaded ────────────────────────────────")
print(f"   Total existing stations : {len(stations):,}")
print(f"   Columns                 : {stations.columns.tolist()}")
print(f"\n   First 5 rows:")
print(stations.head(5).to_string(index=False))


# ============================================================
# STEP 3 — Haversine Distance Formula (manual implementation)
# Computes the great-circle distance (km) between two points
# on Earth given their latitude and longitude in degrees.
#
# Formula:
#   a = sin²(Δlat/2) + cos(lat1) · cos(lat2) · sin²(Δlon/2)
#   distance = 2R · arcsin(√a)      where R = 6371 km
# ============================================================

def haversine(lat1, lon1, lat2_arr, lon2_arr):
    """
    Vectorised Haversine distance from one point to an array
    of points.  All inputs in decimal degrees.
    Returns a NumPy array of distances in kilometres.

    Parameters
    ----------
    lat1, lon1   : float  — single query point
    lat2_arr     : array  — station latitudes
    lon2_arr     : array  — station longitudes
    """
    R = 6371.0                                  # Earth radius (km)

    # Convert degrees → radians
    lat1_r  = np.radians(lat1)
    lon1_r  = np.radians(lon1)
    lat2_r  = np.radians(lat2_arr)
    lon2_r  = np.radians(lon2_arr)

    # Differences
    d_lat = lat2_r - lat1_r
    d_lon = lon2_r - lon1_r

    # Haversine core
    a = (np.sin(d_lat / 2.0) ** 2
         + np.cos(lat1_r) * np.cos(lat2_r) * np.sin(d_lon / 2.0) ** 2)

    return R * 2.0 * np.arcsin(np.sqrt(a))


# ============================================================
# STEP 4 — Compute nearest station for every Top-15 location
# ============================================================

# Pre-extract station coordinate arrays (avoids repeated
# DataFrame lookups inside the loop)
st_lat = stations["latitude"].values
st_lon = stations["longitude"].values
st_name = stations["station_name"].values

nearest_names   = []
nearest_dists   = []

print(f"\n── Computing nearest-station distances ────────────────")

for _, row in top15.iterrows():
    # Distance from this traffic sensor to every station
    dists = haversine(row["LATITUDE"], row["LONGITUDE"],
                      st_lat, st_lon)

    # Index of the closest station
    idx = np.argmin(dists)

    nearest_names.append(st_name[idx])
    nearest_dists.append(round(dists[idx], 3))

    print(f"   {row['GEOHASH']:10s}  →  "
          f"{st_name[idx]:30s}  {dists[idx]:.3f} km")


# ============================================================
# STEP 5 — Add new columns to top15
# ============================================================

top15 = top15.copy()                           # avoid SettingWithCopyWarning
top15["nearest_station_name"] = nearest_names
top15["nearest_distance_km"]  = nearest_dists


# ============================================================
# STEP 6 — Classify competition level
# ============================================================

def classify_competition(dist_km):
    """
    Assign a competition level based on distance to the
    nearest existing gas station.

      < 0.5 km   →  High Competition
      0.5–1.0 km →  Moderate Competition
      > 1.0 km   →  Low Competition
    """
    if dist_km < 0.5:
        return "High Competition"
    elif dist_km <= 1.0:
        return "Moderate Competition"
    else:
        return "Low Competition"

top15["competition_level"] = top15["nearest_distance_km"].apply(
    classify_competition
)


# ============================================================
# STEP 7 — Print updated Top-15 table
# ============================================================

print(f"\n── Top 15 Locations — Competition Analysis ────────────")

display_cols = [
    "DemandRank", "GEOHASH", "LATITUDE", "LONGITUDE",
    "DemandScore", "nearest_station_name",
    "nearest_distance_km", "competition_level"
]

pd.set_option("display.max_colwidth", 30)
print(top15[display_cols].to_string(index=False))

# Summary counts per competition level
print(f"\n── Competition Level Breakdown ────────────────────────")
level_counts = top15["competition_level"].value_counts()
for level, count in level_counts.items():
    print(f"   {level:25s} : {count} location(s)")


# ============================================================
# STEP 8 — Horizontal bar chart: nearest distance by location
# ============================================================

# Build location labels for the y-axis
top15["label"] = (
    top15["GEOHASH"] + "\n"
    + top15["LATITUDE"].round(3).astype(str) + "°N, "
    + top15["LONGITUDE"].round(3).astype(str) + "°E"
)

# Assign bar colour based on competition level
COLOR_MAP = {
    "High Competition"     : "#E53935",   # red
    "Moderate Competition" : "#FB8C00",   # orange
    "Low Competition"      : "#43A047",   # green
}
bar_colors = top15["competition_level"].map(COLOR_MAP).values

fig, ax = plt.subplots(figsize=(13, 8))

# Plot bars — reversed so rank 1 appears at the top
bars = ax.barh(
    range(15),
    top15["nearest_distance_km"].values[::-1],
    color=bar_colors[::-1],
    edgecolor="white",
    linewidth=0.7,
    height=0.65,
)

# Distance labels at the end of each bar
for bar, dist, level in zip(
        bars,
        top15["nearest_distance_km"].values[::-1],
        top15["competition_level"].values[::-1]):
    ax.text(
        bar.get_width() + 0.015,
        bar.get_y() + bar.get_height() / 2,
        f"{dist:.3f} km",
        va="center", ha="left",
        fontsize=8.5, color="#333333", fontweight="bold"
    )

# Y-axis labels
ax.set_yticks(range(15))
ax.set_yticklabels(top15["label"].values[::-1], fontsize=8.5)

# Reference lines for competition thresholds
ax.axvline(0.5, color="#E53935", linestyle="--",
           linewidth=1.2, alpha=0.6, label="0.5 km threshold")
ax.axvline(1.0, color="#FB8C00", linestyle="--",
           linewidth=1.2, alpha=0.6, label="1.0 km threshold")

# Titles and axis labels
ax.set_title(
    "Nearest Existing Gas Station Distance — Top 15 Demand Locations\n"
    "Istanbul Gas Station Site Selection — October 2024",
    fontsize=14, fontweight="bold", pad=14
)
ax.set_xlabel("Distance to Nearest Existing Station (km)", fontsize=11)
ax.set_ylabel("Location (Geohash  |  Coordinates)", fontsize=11)
ax.set_xlim(0, top15["nearest_distance_km"].max() * 1.22)
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))

# Legend — threshold lines + competition colours
line_handles = [
    plt.Line2D([0], [0], color="#E53935", linestyle="--",
               linewidth=1.5, label="0.5 km (High/Moderate boundary)"),
    plt.Line2D([0], [0], color="#FB8C00", linestyle="--",
               linewidth=1.5, label="1.0 km (Moderate/Low boundary)"),
]
patch_handles = [
    mpatches.Patch(color=COLOR_MAP[lv], label=lv)
    for lv in ["High Competition", "Moderate Competition", "Low Competition"]
]
ax.legend(
    handles=line_handles + patch_handles,
    fontsize=8.5, loc="lower right",
    framealpha=0.9, edgecolor="#CCCCCC"
)

plt.tight_layout()
plt.savefig("plot_competition_distance.png", dpi=150)
plt.show()
print("\nSaved → plot_competition_distance.png")


# ============================================================
# STEP 9 — Summary statistics of nearest_distance_km
# ============================================================

print(f"\n── nearest_distance_km — Summary Statistics ───────────")
stats = top15["nearest_distance_km"].describe()
for stat_name, val in stats.items():
    print(f"   {stat_name:8s} : {val:.3f} km")

print(f"\n   Closest  station : "
      f"{top15.loc[top15['nearest_distance_km'].idxmin(), 'GEOHASH']}  "
      f"({top15['nearest_distance_km'].min():.3f} km)")
print(f"   Farthest station : "
      f"{top15.loc[top15['nearest_distance_km'].idxmax(), 'GEOHASH']}  "
      f"({top15['nearest_distance_km'].max():.3f} km)")


# ============================================================
# DONE
# ============================================================
print("\nSection 3 complete — Competition analysis finished.")

SECTION 3 — Competition & Distance Analysis

── Gas Stations Loaded ────────────────────────────────
   Total existing stations : 788
   Columns                 : ['station_name', 'latitude', 'longitude']

   First 5 rows:
station_name  latitude  longitude
     Aytemiz 40.896837  29.224951
        Opet 41.026080  28.979786
Petrol Ofisi 41.028838  29.017440
Petrol Ofisi 41.114194  29.059922
Class Petrol 41.000190  28.800738

── Computing nearest-station distances ────────────────
   sxkb6p      →  Türkiye Petrolleri              1.000 km
   sxk9ku      →  Opet Üsküdar                    0.224 km
   sxk9s5      →  Shell                           0.335 km
   sxk9sh      →  Shell                           0.491 km
   sxk9vb      →  Opet Beykoz                     0.521 km
   sxk9kg      →  Total                           0.227 km
   sxk9m5      →  Opet                            0.458 km
   sxk9s6      →  Shell                           0.839 km
   sxk9ub      →  Shell                     

Data Source:
Existing gas station locations were obtained from OpenStreetMap using the Overpass API.

Query Used:
amenity=fuel within the administrative boundary of Istanbul.

Collection Method:
The query was executed via Overpass Turbo, and the results were exported as CSV.
Latitude and longitude coordinates were extracted from the @lat and @lon fields.

Total Stations Collected: 788

SECTION 4:

In [32]:
# ============================================================
# SECTION 4 — Final Site Selection
# Istanbul Gas Station Investment Decision
# ============================================================
# Prerequisites (must have run in the same session):
#   Section 1  →  `summary` dataframe
#   Section 2  →  `DemandScore`, `DemandRank` columns in top15
#   Section 3  →  `nearest_distance_km`, `competition_level`
#                  columns added to `top15`
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches


# ============================================================
# STEP 1 — Min-Max normalise DemandScore & nearest_distance_km
# Both inputs must be on the same 0-1 scale before weighting.
# For distance: a higher value means LESS competition nearby,
# which is BETTER — so we normalise it directly (no inversion).
# ============================================================

def assign_district(row):
    # Example hardcoded rules LATITUDE
    if (row["LATITUDE"] > 40.8664 and row["LATITUDE"] < 40.8666) and (row["LONGITUDE"] > 29.2730 and row["LONGITUDE"] < 29.2732):
        return "Pendik"
    elif row["LATITUDE"] == 41.0257 and row["LONGITUDE"] == 29.0424:
        return "Uskudar"
    elif row["LATITUDE"] == 41.0696 and row["LONGITUDE"] == 29.0094:
        return "Besiktas"
    else:
        return "Unknownnn"

def minmax_norm(series):
    """Normalise a pandas Series to the range [0, 1]."""
    col_min = series.min()
    col_max = series.max()
    return (series - col_min) / (col_max - col_min)

top15 = top15.copy()                  # avoid SettingWithCopyWarning

top15["norm_DemandScore"]         = minmax_norm(top15["DemandScore"])
top15["norm_nearest_distance_km"] = minmax_norm(top15["nearest_distance_km"])

print("=" * 60)
print("SECTION 4 — Final Site Selection")
print("=" * 60)

print("\n── Normalised inputs ──────────────────────────────────")
print(top15[["GEOHASH", "norm_DemandScore",
             "norm_nearest_distance_km"]].round(4).to_string(index=False))


# ============================================================
# STEP 2 — Compute FinalScore
#
# FinalScore = 0.60 * norm_DemandScore
#            + 0.40 * norm_nearest_distance_km
#
# Weight logic:
#   60 % — traffic demand is the primary business driver
#   40 % — distance from competition determines market gap
# ============================================================

top15["FinalScore"] = (
    0.60 * top15["norm_DemandScore"] +
    0.40 * top15["norm_nearest_distance_km"]
)

print("\n── FinalScore computed ────────────────────────────────")
print(top15[["GEOHASH", "DemandScore", "nearest_distance_km",
             "FinalScore"]].round(4).to_string(index=False))


# ============================================================
# STEP 3 — Rank all locations by FinalScore (descending)
# ============================================================

top15["FinalRank"] = (
    top15["FinalScore"]
    .rank(ascending=False, method="min")
    .astype(int)
)


# ============================================================
# STEP 4 — Select exactly the Top 3 locations
# ============================================================

top3 = (
    top15
    .nsmallest(3, "FinalRank")
    .reset_index(drop=True)
)
top3.index += 1                       # display index 1, 2, 3
top3["district"] = top3.apply(assign_district, axis=1)

# ============================================================
# STEP 5 & 6 — Print the professional summary table
# ============================================================

REPORT_COLS = [
    "FinalRank", "GEOHASH",
    "LATITUDE", "LONGITUDE",
    "DemandScore", "nearest_distance_km",
    "competition_level", "FinalScore", "district"
]

print("\n" + "=" * 60)
print("  TOP 3 PROPOSED GAS STATION LOCATIONS")
print("  Istanbul — October 2024 Traffic Analysis")
print("=" * 60)
print(top3[REPORT_COLS].round(4).to_string(index=False))
print("=" * 60)


# ============================================================
# STEP 7 — Horizontal bar chart: Top 10 by FinalScore
# ============================================================

# Take top 10 for the chart (sorted ascending so best site
# appears at the top of the horizontal chart)
top10_plot = (
    top15
    .nsmallest(10, "FinalRank")
    .sort_values("FinalRank", ascending=False)   # reversed for barh
    .reset_index(drop=True)
)

# Build y-axis labels
top10_plot["label"] = (
    top10_plot["GEOHASH"] + "\n"
    + top10_plot["LATITUDE"].round(3).astype(str)  + "°N, "
    + top10_plot["LONGITUDE"].round(3).astype(str) + "°E"
)

# Colour: top-3 highlighted in deep blue, rest in steel blue
bar_colors = [
    "#1565C0" if rank <= 3 else "#90CAF9"
    for rank in top10_plot["FinalRank"]
]

fig, ax = plt.subplots(figsize=(13, 8))

bars = ax.barh(
    range(10),
    top10_plot["FinalScore"].values,
    color=bar_colors,
    edgecolor="white",
    linewidth=0.7,
    height=0.65,
)

# Score labels at end of each bar
for bar, score, rank in zip(bars,
                             top10_plot["FinalScore"].values,
                             top10_plot["FinalRank"].values):
    label_text = f"  {score:.4f}"
    if rank <= 3:
        label_text += f"  ← Rank #{rank}"
    ax.text(
        bar.get_width() + 0.005,
        bar.get_y() + bar.get_height() / 2,
        label_text,
        va="center", ha="left",
        fontsize=8.5,
        color="#1565C0" if rank <= 3 else "#555555",
        fontweight="bold" if rank <= 3 else "normal"
    )

# Y-axis labels
ax.set_yticks(range(10))
ax.set_yticklabels(top10_plot["label"].values, fontsize=8.5)

# Vertical reference line at the Top-3 cutoff score
cutoff = top3["FinalScore"].min()
ax.axvline(
    cutoff, color="#1565C0", linestyle="--",
    linewidth=1.3, alpha=0.65, label=f"Top-3 cutoff ({cutoff:.4f})"
)

ax.set_title(
    "Top 10 Locations by Final Investment Score\n"
    "Istanbul Gas Station Site Selection — October 2024",
    fontsize=14, fontweight="bold", pad=14
)
ax.set_xlabel(
    "Final Score  (0.60 × Demand  +  0.40 × Distance from Competition)",
    fontsize=11
)
ax.set_ylabel("Location (Geohash  |  Coordinates)", fontsize=11)
ax.set_xlim(0, top10_plot["FinalScore"].max() * 1.25)
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))

# Legend — colour key + cutoff line
legend_handles = [
    mpatches.Patch(color="#1565C0", label="Top 3 selected sites"),
    mpatches.Patch(color="#90CAF9", label="Other top-10 candidates"),
    plt.Line2D([0], [0], color="#1565C0", linestyle="--",
               linewidth=1.5, label=f"Top-3 cutoff ({cutoff:.4f})")
]
ax.legend(handles=legend_handles, fontsize=9,
          loc="lower right", framealpha=0.9, edgecolor="#CCCCCC")

# Weight annotation box
weight_note = (
    "FinalScore weights\n"
    "Demand score          : 0.60\n"
    "Distance from competition : 0.40"
)
ax.text(
    0.99, 0.98, weight_note,
    transform=ax.transAxes,
    fontsize=8, va="top", ha="right",
    bbox=dict(boxstyle="round,pad=0.45", fc="white",
              ec="#BBBBBB", alpha=0.88)
)

plt.tight_layout()
plt.savefig("plot_final_site_selection.png", dpi=150)
plt.show()
print("\nSaved → plot_final_site_selection.png")


# ============================================================
# STEP 8 — Per-location investment justifications
# ============================================================

# Colour label maps for printing
COMP_EMOJI = {
    "Low Competition"      : "🟢",
    "Moderate Competition" : "🟠",
    "High Competition"     : "🔴",
}

print("\n" + "=" * 60)
print("  INVESTMENT JUSTIFICATIONS — TOP 3 SITES")
print("=" * 60)

for i, (_, row) in enumerate(top3.iterrows(), start=1):

    comp_icon  = COMP_EMOJI.get(row["competition_level"], "")
    comp_label = row["competition_level"]
    dist_km    = row["nearest_distance_km"]
    district = row["district"]
    demand     = row["DemandScore"]
    final      = row["FinalScore"]
    geohash    = row["GEOHASH"]
    lat        = row["LATITUDE"]
    lon        = row["LONGITUDE"]



    # Build demand description tier
    if demand >= 0.50:
        demand_desc = "exceptionally high"
    elif demand >= 0.35:
        demand_desc = "high"
    else:
        demand_desc = "solid"

    # Build distance description tier
    if dist_km > 1.5:
        dist_desc = "well over 1.5 km from any rival"
    elif dist_km > 1.0:
        dist_desc = f"{dist_km:.3f} km from the nearest competitor"
    elif dist_km > 0.5:
        dist_desc = (f"{dist_km:.3f} km from the nearest station "
                     f"— within moderate proximity")
    else:
        dist_desc = (f"only {dist_km:.3f} km from an existing station "
                     f"— entering a crowded micro-market")

    justification = (
        f"Location {i} ({geohash}, {lat:.4f}°N {lon:.4f}°E) records a "
        f"{demand_desc} demand score of {demand:.4f}, driven by consistent "
        f"high vehicle throughput on this corridor. "
        f"The nearest existing station is {dist_desc}, "
        f"placing it in the '{comp_label}' category {comp_icon}. "
        f"With a combined FinalScore of {final:.4f}, this site ranks "
        f"#{i} overall and "
        + (
            "represents an outstanding greenfield opportunity with minimal "
            "competitive pressure."
            if comp_label == "Low Competition" else
            "offers a viable opportunity where demand substantially "
            "outweighs the moderate competitive presence."
            if comp_label == "Moderate Competition" else
            "warrants careful evaluation — strong demand exists but "
            "differentiation strategy will be essential to compete."
        )
    )

    print(f"\nLocation {i} Justification:")
    print(f"  {justification}")

print()


# ============================================================
# DONE
# ============================================================
print("Section 4 complete — Final locations proposed.")

SECTION 4 — Final Site Selection

── Normalised inputs ──────────────────────────────────
GEOHASH  norm_DemandScore  norm_nearest_distance_km
 sxkb6p            1.0000                    0.4288
 sxk9ku            0.8957                    0.0284
 sxk9s5            0.7139                    0.0857
 sxk9sh            0.7051                    0.1662
 sxk9vb            0.6117                    0.1816
 sxk9kg            0.4353                    0.0299
 sxk9m5            0.1771                    0.1491
 sxk9s6            0.1423                    0.3457
 sxk9ub            0.1184                    1.0000
 sxk9u8            0.1018                    0.5428
 sxk9n3            0.0917                    0.0722
 sxk9sk            0.0788                    0.1357
 sxk9pq            0.0526                    0.2895
 sxk3k2            0.0303                    0.0805
 sxkb97            0.0000                    0.0000

── FinalScore computed ────────────────────────────────
GEOHASH  DemandScore 

SECTION 5:

In [25]:
# ============================================================
# SECTION 5 — Visualizations
# Istanbul Gas Station Site Selection Analysis
# ============================================================
# Plot 1  — Interactive Demand Score Map         (Folium HTML)
# Plot 2  — Interactive Competition Map          (Folium HTML)
# Plot 3  — Investment Scorecard — Top-3 Sites   (matplotlib PNG)
# Plot 4  — FinalScore Ranking — Top-15 Context  (matplotlib PNG)
#
# Install once if needed:
#   pip install folium branca
#
# Prerequisites (must already exist in session):
#   `summary`  — LATITUDE, LONGITUDE, DemandScore
#   `top15`    — all scoring columns including FinalScore, FinalRank
#   `top3`     — 3 proposed sites with all metrics
#   `stations` — station_name, latitude, longitude
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import folium
from folium import FeatureGroup, LayerControl, CircleMarker, Marker
import branca.colormap as cm

print("=" * 60)
print("SECTION 5 — Visualizations")
print("=" * 60)


# ============================================================
# SHARED CONFIGURATION
# ============================================================

# Colour maps used across multiple plots
COMP_COLORS = {
    "Low Competition"      : "#43A047",
    "Moderate Competition" : "#FB8C00",
    "High Competition"     : "#E53935",
}

# Map centre: geographic mean of all sensor locations
MAP_CENTER  = [summary["LATITUDE"].mean(), summary["LONGITUDE"].mean()]
ZOOM_START  = 11

# YlOrRd continuous colormap scaled to the actual DemandScore range
demand_colormap = cm.LinearColormap(
    colors=["#ffffb2", "#fecc5c", "#fd8d3c", "#f03b20", "#bd0026"],
    vmin=summary["DemandScore"].min(),
    vmax=summary["DemandScore"].max(),
    caption="Demand Score  (0 = lowest  →  1 = highest)"
)

# Distinct border colours for each proposed site's competition circle
SITE_CIRCLE_COLORS = ["#1565C0", "#7B1FA2", "#E65100"]


# ============================================================
# PLOT 1 — Interactive Demand Score Map
# ------------------------------------------------------------
# Every sensor location is drawn as a small CircleMarker
# coloured by its DemandScore on the YlOrRd scale.
# The 3 proposed sites are overlaid as prominent red star
# markers with rich popup cards.
# Saved as: interactive_s5_demand_map.html
# ============================================================

print("\n── Plot 1: Interactive Demand Score Map ────────────────")

m1 = folium.Map(
    location=MAP_CENTER,
    zoom_start=ZOOM_START,
    tiles="CartoDB positron",       # clean light basemap — roads,
                                    # bridges and Bosphorus clearly visible
    control_scale=True
)

# ── Layer 1: all 2,439 sensor demand locations ────────────────
layer_sensors = FeatureGroup(
    name="Demand Sensors  (2,439 locations)", show=True
)

for _, row in summary.iterrows():
    colour = demand_colormap(row["DemandScore"])
    CircleMarker(
        location=[row["LATITUDE"], row["LONGITUDE"]],
        radius=4,
        color=colour,
        fill=True,
        fill_color=colour,
        fill_opacity=0.80,
        weight=0.4,
        tooltip=folium.Tooltip(
            f"<b>Demand Score:</b> {row['DemandScore']:.3f}<br>"
            f"<b>Lat:</b> {row['LATITUDE']:.5f}°N<br>"
            f"<b>Lon:</b> {row['LONGITUDE']:.5f}°E",
            sticky=True
        )
    ).add_to(layer_sensors)

layer_sensors.add_to(m1)

# ── Layer 2: 3 proposed sites as star markers ─────────────────
layer_proposed_m1 = FeatureGroup(
    name="Proposed New Sites  (Top 3)", show=True
)

for _, row in top3.iterrows():
    comp_color = COMP_COLORS.get(row["competition_level"], "#607D8B")
    popup_html = f"""
    <div style="font-family:Arial,sans-serif;font-size:13px;
                min-width:220px;line-height:1.7;">
      <div style="background:#1a1a2e;color:white;padding:8px 12px;
                  border-radius:6px 6px 0 0;font-size:15px;
                  font-weight:bold;letter-spacing:0.5px;">
        ★ Proposed Site #{int(row['FinalRank'])}
      </div>
      <div style="padding:10px 12px;border:1px solid #ddd;
                  border-top:none;border-radius:0 0 6px 6px;">
        <table style="width:100%;border-collapse:collapse;">
          <tr><td style="color:#555;padding:2px 0;">Geohash</td>
              <td style="font-weight:bold;text-align:right;">{row['GEOHASH']}</td></tr>
          <tr><td style="color:#555;padding:2px 0;">Demand Score</td>
              <td style="font-weight:bold;text-align:right;color:#f03b20;">
                {row['DemandScore']:.4f}</td></tr>
          <tr><td style="color:#555;padding:2px 0;">Final Score</td>
              <td style="font-weight:bold;text-align:right;color:#1565C0;">
                {row['FinalScore']:.4f}</td></tr>
          <tr><td style="color:#555;padding:2px 0;">Nearest Station</td>
              <td style="font-weight:bold;text-align:right;">
                {row['nearest_distance_km']:.3f} km</td></tr>
          <tr><td style="color:#555;padding:2px 0;">Competition</td>
              <td style="text-align:right;">
                <span style="background:{comp_color};color:white;
                             padding:2px 8px;border-radius:12px;
                             font-size:11px;font-weight:bold;">
                  {row['competition_level']}
                </span></td></tr>
          <tr><td style="color:#555;padding:2px 0;">Coordinates</td>
              <td style="font-size:11px;text-align:right;color:#666;">
                {row['LATITUDE']:.5f}°N, {row['LONGITUDE']:.5f}°E</td></tr>
        </table>
      </div>
    </div>"""
    Marker(
        location=[row["LATITUDE"], row["LONGITUDE"]],
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=folium.Tooltip(
            f"<b>★ Proposed Site #{int(row['FinalRank'])}</b><br>"
            f"Final Score: {row['FinalScore']:.4f}",
            sticky=True
        ),
        icon=folium.Icon(color="red", icon_color="white",
                         icon="star", prefix="fa")
    ).add_to(layer_proposed_m1)

layer_proposed_m1.add_to(m1)

# ── Colormap legend + layer control + title ───────────────────
demand_colormap.add_to(m1)
LayerControl(collapsed=False).add_to(m1)
m1.get_root().html.add_child(folium.Element("""
<div style="position:fixed;top:12px;left:50%;transform:translateX(-50%);
            z-index:9999;background:rgba(255,255,255,0.93);
            padding:10px 22px;border-radius:8px;border:1px solid #ccc;
            text-align:center;font-family:Arial,sans-serif;
            box-shadow:0 2px 8px rgba(0,0,0,.15);">
  <div style="font-size:16px;font-weight:bold;color:#1a1a2e;">
    Traffic Demand Score Distribution — Istanbul
  </div>
  <div style="font-size:11px;color:#555;margin-top:3px;">
    2,439 sensor locations  |  October 2024  |  3 proposed sites marked ★
  </div>
</div>"""))

m1.save("interactive_s5_demand_map.html")
print("   Saved → interactive_s5_demand_map.html")


# ============================================================
# PLOT 2 — Interactive Competition Map
# ------------------------------------------------------------
# Existing 788 gas stations appear as small blue dots.
# The 3 proposed sites are large red star markers.
# Around each proposed site a transparent circle shows the
# exact distance to the nearest competitor, visually proving
# the competition gap that justifies each investment decision.
# Saved as: interactive_s5_competition_map.html
# ============================================================

print("\n── Plot 2: Interactive Competition Map ─────────────────")

m2 = folium.Map(
    location=MAP_CENTER,
    zoom_start=ZOOM_START,
    tiles="CartoDB positron",
    control_scale=True
)

# ── Layer 1: existing 788 gas stations ───────────────────────
layer_existing = FeatureGroup(
    name=f"Existing Gas Stations  (n = {len(stations):,})", show=True
)

for _, row in stations.iterrows():
    CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color="#1565C0",
        fill=True,
        fill_color="#1565C0",
        fill_opacity=0.55,
        weight=0.5,
        tooltip=folium.Tooltip(
            f"<b>{row['station_name']}</b><br>"
            f"{row['latitude']:.5f}°N, {row['longitude']:.5f}°E",
            sticky=True
        )
    ).add_to(layer_existing)

layer_existing.add_to(m2)

# ── Layer 2: competition-gap circles ─────────────────────────
layer_circles = FeatureGroup(
    name="Distance to Nearest Competitor  (circles)", show=True
)

for i, (_, row) in enumerate(top3.iterrows()):
    circle_color = SITE_CIRCLE_COLORS[i % len(SITE_CIRCLE_COLORS)]
    folium.Circle(
        location=[row["LATITUDE"], row["LONGITUDE"]],
        radius=row["nearest_distance_km"] * 1000,   # convert km → metres
        color=circle_color,
        weight=2.0,
        fill=True,
        fill_color=circle_color,
        fill_opacity=0.08,
        tooltip=folium.Tooltip(
            f"<b>Site #{int(row['FinalRank'])} — competition radius</b><br>"
            f"Nearest station: {row['nearest_distance_km']:.3f} km",
            sticky=True
        )
    ).add_to(layer_circles)

layer_circles.add_to(m2)

# ── Layer 3: 3 proposed sites as star markers ─────────────────
layer_proposed_m2 = FeatureGroup(
    name="Proposed New Stations  (n = 3)", show=True
)

for i, (_, row) in enumerate(top3.iterrows()):
    comp_color   = COMP_COLORS.get(row["competition_level"], "#607D8B")
    circle_color = SITE_CIRCLE_COLORS[i % len(SITE_CIRCLE_COLORS)]
    popup_html_m2 = f"""
    <div style="font-family:Arial,sans-serif;font-size:13px;
                min-width:230px;line-height:1.7;">
      <div style="background:#b71c1c;color:white;padding:8px 12px;
                  border-radius:6px 6px 0 0;font-size:15px;
                  font-weight:bold;letter-spacing:0.5px;">
        ★ Proposed Site #{int(row['FinalRank'])}
      </div>
      <div style="padding:10px 12px;border:1px solid #ddd;
                  border-top:none;border-radius:0 0 6px 6px;">
        <table style="width:100%;border-collapse:collapse;">
          <tr><td style="color:#555;padding:3px 0;">Geohash</td>
              <td style="font-weight:bold;text-align:right;">{row['GEOHASH']}</td></tr>
          <tr><td style="color:#555;padding:3px 0;">Demand Score</td>
              <td style="font-weight:bold;text-align:right;color:#f03b20;">
                {row['DemandScore']:.4f}</td></tr>
          <tr><td style="color:#555;padding:3px 0;">Nearest Station</td>
              <td style="font-weight:bold;text-align:right;">
                {row['nearest_distance_km']:.3f} km</td></tr>
          <tr><td style="color:#555;padding:3px 0;">Competition</td>
              <td style="text-align:right;">
                <span style="background:{comp_color};color:white;
                             padding:2px 8px;border-radius:12px;
                             font-size:11px;font-weight:bold;">
                  {row['competition_level']}
                </span></td></tr>
          <tr><td style="color:#555;padding:3px 0;">Coordinates</td>
              <td style="font-size:11px;text-align:right;color:#666;">
                {row['LATITUDE']:.5f}°N,<br>{row['LONGITUDE']:.5f}°E</td></tr>
        </table>
        <div style="margin-top:8px;padding:6px 8px;background:#f5f5f5;
                    border-radius:4px;font-size:11px;color:#444;
                    border-left:3px solid {circle_color};">
          The circle shows the {row['nearest_distance_km']:.3f} km gap to the
          nearest rival — confirming
          <b>{row['competition_level'].lower()}</b> in this micro-market.
        </div>
      </div>
    </div>"""
    Marker(
        location=[row["LATITUDE"], row["LONGITUDE"]],
        popup=folium.Popup(popup_html_m2, max_width=290),
        tooltip=folium.Tooltip(
            f"<b>★ Proposed Site #{int(row['FinalRank'])}</b><br>"
            f"Nearest rival: {row['nearest_distance_km']:.3f} km<br>"
            f"{row['competition_level']}",
            sticky=True
        ),
        icon=folium.Icon(color="red", icon_color="white",
                         icon="star", prefix="fa")
    ).add_to(layer_proposed_m2)

layer_proposed_m2.add_to(m2)

# ── Legend panel + layer control + title ─────────────────────
m2.get_root().html.add_child(folium.Element("""
<div style="position:fixed;bottom:28px;right:12px;z-index:9999;
            background:rgba(255,255,255,0.95);padding:14px 18px;
            border-radius:8px;border:1px solid #ccc;
            font-family:Arial,sans-serif;font-size:13px;
            box-shadow:0 2px 8px rgba(0,0,0,.15);min-width:210px;">
  <div style="font-weight:bold;font-size:14px;margin-bottom:10px;
              color:#1a1a2e;border-bottom:1px solid #eee;padding-bottom:6px;">
    Map Legend</div>
  <div style="display:flex;align-items:center;margin-bottom:7px;">
    <span style="display:inline-block;width:14px;height:14px;background:#1565C0;
                 border-radius:50%;margin-right:9px;opacity:0.75;"></span>
    Existing gas stations</div>
  <div style="display:flex;align-items:center;margin-bottom:7px;">
    <span style="font-size:18px;color:#D32F2F;margin-right:9px;line-height:1;">
      ★</span>Proposed new site</div>
  <div style="display:flex;align-items:center;margin-bottom:2px;">
    <span style="display:inline-block;width:18px;height:0;
                 border:2px dashed #555;margin-right:9px;opacity:0.6;"></span>
    Distance to nearest rival</div>
  <div style="margin-top:10px;padding-top:8px;border-top:1px solid #eee;
              font-size:11px;color:#666;line-height:1.5;">
    Circle radius = nearest competitor<br>distance in metres.<br>
    Larger circle → less competition.</div>
</div>"""))

m2.get_root().html.add_child(folium.Element("""
<div style="position:fixed;top:12px;left:50%;transform:translateX(-50%);
            z-index:9999;background:rgba(255,255,255,0.93);
            padding:10px 22px;border-radius:8px;border:1px solid #ccc;
            text-align:center;font-family:Arial,sans-serif;
            box-shadow:0 2px 8px rgba(0,0,0,.15);">
  <div style="font-size:16px;font-weight:bold;color:#1a1a2e;">
    Existing Gas Stations vs. Proposed New Locations — Istanbul
  </div>
  <div style="font-size:11px;color:#555;margin-top:3px;">
    788 existing stations  |  3 proposed sites  |
    Circles show gap to nearest competitor
  </div>
</div>"""))

LayerControl(collapsed=False).add_to(m2)
m2.save("interactive_s5_competition_map.html")
print("   Saved → interactive_s5_competition_map.html")


# ============================================================
# PLOT 3 — Hourly Traffic Pattern at Proposed Locations
# ------------------------------------------------------------
# Filters traffic_df to the 3 proposed sites using exact
# (LATITUDE, LONGITUDE) tuple matching — avoids any float-
# precision issues.  Computes mean vehicle count per hour of
# day for each site, then plots three lines on a single axis.
# Reveals whether demand is consistent across the full day
# or concentrated in narrow peaks.
# Saved as: plot_s5_hourly_traffic_pattern.png
# ============================================================

print("\n── Plot 3: Hourly Traffic Pattern — Proposed Sites ─────")

# ── Step 1: Filter traffic_df to the 3 proposed sites ────────
# Build a set of exact (lat, lon) tuples from top3, then use
# pandas isin on a zipped Series — fully vectorised, no loops.
site_coords  = list(zip(top3["LATITUDE"], top3["LONGITUDE"]))
coord_series = list(zip(df["LATITUDE"], df["LONGITUDE"]))
site_traffic = df[
    pd.Series(coord_series).isin(site_coords).values
].copy()

# ── Step 2: Ensure HOUR column exists ────────────────────────
if "HOUR" not in site_traffic.columns:
    site_traffic["HOUR"] = site_traffic["DATE_TIME"].dt.hour

# ── Step 3: Aggregate — mean vehicles per location per hour ──
hourly_profile = (
    site_traffic
    .groupby(["LATITUDE", "LONGITUDE", "HOUR"])["NUMBER_OF_VEHICLES"]
    .mean()
    .reset_index()
    .sort_values(["LATITUDE", "LONGITUDE", "HOUR"])
)

# ── Step 4: Build the time-series plot ───────────────────────
LINE_COLORS  = ["#E53935", "#1565C0", "#2E7D32"]   # red, blue, green
LINE_MARKERS = ["o", "s", "^"]

fig, ax = plt.subplots(figsize=(13, 6))

for i, (_, site_row) in enumerate(top3.sort_values("FinalRank").iterrows()):
    lat  = site_row["LATITUDE"]
    lon  = site_row["LONGITUDE"]
    rank = int(site_row["FinalRank"])
    gh   = site_row["GEOHASH"]

    data = hourly_profile[
        (hourly_profile["LATITUDE"]  == lat) &
        (hourly_profile["LONGITUDE"] == lon)
    ]

    ax.plot(
        data["HOUR"],
        data["NUMBER_OF_VEHICLES"],
        color=LINE_COLORS[i],
        marker=LINE_MARKERS[i],
        linewidth=2.2,
        markersize=5,
        label=f"Site #{rank} — {gh}"
    )

# ── Formatting ────────────────────────────────────────────────
ax.set_xticks(range(0, 24))
ax.set_title(
    "Average Hourly Traffic Pattern — Proposed Gas Station Locations\n"
    "Consistency of Traffic Volume Across 24 Hours",
    fontsize=14, fontweight="bold", pad=14
)
ax.set_xlabel("Hour of Day", fontsize=11)
ax.set_ylabel("Average Number of Vehicles", fontsize=11)
ax.tick_params(axis="both", labelsize=9)
ax.grid(axis="both", linestyle="--", alpha=0.4)
ax.legend(fontsize=10, loc="upper right",
          framealpha=0.9, edgecolor="#AAAAAA")
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{int(x):,}")
)

plt.tight_layout()
plt.savefig("plot_s5_hourly_traffic_pattern.png",
            dpi=300, bbox_inches="tight")
plt.show()
print("   Saved → plot_s5_hourly_traffic_pattern.png")


# ============================================================
# PLOT 4 — FinalScore Ranking: Full Top-15 Context
# ------------------------------------------------------------
# Shows where the top-3 selected sites sit within the broader
# top-15 candidate pool.  Bars are coloured by competition
# level and the selected trio carries a bold black border.
# Saved as: plot_s5_finalscore_ranking.png
# ============================================================

print("\n── Plot 4: FinalScore Ranking — Top-15 Context ─────────")

top15_sorted = (top15.sort_values("FinalScore", ascending=True)
                     .reset_index(drop=True))
top15_sorted["label"] = (
    top15_sorted["GEOHASH"] + "  |  "
    + top15_sorted["LATITUDE"].round(3).astype(str) + "°N"
)

bar_colors_15 = [COMP_COLORS[lvl] for lvl in top15_sorted["competition_level"]]
edge_colors   = ["black" if r <= 3 else "white" for r in top15_sorted["FinalRank"]]
edge_widths   = [1.5    if r <= 3 else 0.5     for r in top15_sorted["FinalRank"]]

fig, ax = plt.subplots(figsize=(13, 9))
bars = ax.barh(range(15), top15_sorted["FinalScore"].values,
               color=bar_colors_15, edgecolor=edge_colors,
               linewidth=edge_widths, height=0.65, zorder=3)

for bar, val, rank in zip(bars, top15_sorted["FinalScore"].values,
                           top15_sorted["FinalRank"].values):
    suffix = "  ← SELECTED" if rank <= 3 else ""
    ax.text(bar.get_width() + 0.006,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}{suffix}", va="center", ha="left",
            fontsize=8.5,
            fontweight="bold" if rank <= 3 else "normal",
            color="black" if rank <= 3 else "#444444")

ax.set_yticks(range(15))
ax.set_yticklabels(top15_sorted["label"].values, fontsize=8.5)

cutoff = top3["FinalScore"].min()
ax.axvline(cutoff, color="black", linestyle="--", linewidth=1.3,
           alpha=0.7, label=f"Selection cutoff — FinalScore ≥ {cutoff:.4f}")

legend_patches = [mpatches.Patch(color=COMP_COLORS[lv], label=lv)
                  for lv in ["Low Competition",
                              "Moderate Competition",
                              "High Competition"]]
selected_patch = mpatches.Patch(facecolor="white", edgecolor="black",
                                 linewidth=1.5,
                                 label="Selected sites (bold border)")
cutoff_line = plt.Line2D([0], [0], color="black", linestyle="--",
                          linewidth=1.3, label=f"Cutoff ({cutoff:.4f})")
ax.legend(handles=legend_patches + [selected_patch, cutoff_line],
          fontsize=9, loc="lower right",
          framealpha=0.9, edgecolor="#AAAAAA")

ax.set_title(
    "Final Investment Score — All 15 Candidate Locations\n"
    "Coloured by Competition Level  |  Bold Border = Selected",
    fontsize=14, fontweight="bold", pad=14)
ax.set_xlabel(
    "Final Score  (0.60 × Demand  +  0.40 × Distance from Competition)",
    fontsize=11)
ax.set_ylabel("Candidate Location", fontsize=11)
ax.set_xlim(0, top15_sorted["FinalScore"].max() * 1.26)
ax.grid(axis="x", linestyle="--", linewidth=0.5, alpha=0.40)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))

plt.tight_layout()
plt.savefig("plot_s5_finalscore_ranking.png", dpi=150)
plt.show()
print("   Saved → plot_s5_finalscore_ranking.png")


# ============================================================
# SECTION 5 SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("  SECTION 5 SUMMARY — All Files Saved")
print("=" * 60)

outputs = [
    ("interactive_s5_demand_map.html",
     "Interactive demand score map  (open in browser)"),
    ("interactive_s5_competition_map.html",
     "Interactive competition map   (open in browser)"),
    ("plot_s5_hourly_traffic_pattern.png",
     "Hourly traffic time-series — 3 proposed sites"),
    ("plot_s5_finalscore_ranking.png",
     "Full top-15 FinalScore ranking"),
]
for fname, desc in outputs:
    print(f"   ✓  {fname}")
    print(f"      {desc}")

print("\n   HTML maps require an internet connection to load map tiles.")
print("   Open them in Chrome, Edge, or Firefox.")
print("\nSection 5 complete — All visualizations exported.")

SECTION 5 — Visualizations

── Plot 1: Interactive Demand Score Map ────────────────
   Saved → interactive_s5_demand_map.html

── Plot 2: Interactive Competition Map ─────────────────
   Saved → interactive_s5_competition_map.html

── Plot 3: Hourly Traffic Pattern — Proposed Sites ─────
   Saved → plot_s5_hourly_traffic_pattern.png

── Plot 4: FinalScore Ranking — Top-15 Context ─────────
   Saved → plot_s5_finalscore_ranking.png

  SECTION 5 SUMMARY — All Files Saved
   ✓  interactive_s5_demand_map.html
      Interactive demand score map  (open in browser)
   ✓  interactive_s5_competition_map.html
      Interactive competition map   (open in browser)
   ✓  plot_s5_hourly_traffic_pattern.png
      Hourly traffic time-series — 3 proposed sites
   ✓  plot_s5_finalscore_ranking.png
      Full top-15 FinalScore ranking

   HTML maps require an internet connection to load map tiles.
   Open them in Chrome, Edge, or Firefox.

Section 5 complete — All visualizations exported.


SECTION 6

In [26]:
# ============================================================
# SECTION 6 — Discussion
# Istanbul Gas Station Site Selection Analysis
# ============================================================

print("=" * 60)
print("SECTION 6 — Discussion")
print("=" * 60)

discussion = """
The three proposed locations all sit along the TEM motorway corridor
near Ümraniye (approximately 41.09°N, 29.06°E), and that concentration
is not a coincidence. This corridor recorded the highest sustained
vehicle throughput in the entire dataset — the top-3 sites averaged
over 10,000 vehicles per day, compared to a city-wide sensor mean of
roughly 475. What separated them from other high-demand candidates was
the competition gap: Sites 1 and 2 sit more than 1.9 km from any
existing station, and Site 3, despite a slightly closer competitor at
0.52 km, still carries the highest raw DemandScore of all 15 candidates
(0.578). The FinalScore formula — which weights demand at 60% and
competition distance at 40% — rewarded exactly this combination. Several
locations ranked higher on demand alone but fell below the cutoff because
they were already surrounded by rival stations within 300–500 metres,
making them viable for competition but unattractive for a new entrant.

That said, traffic volume is only one piece of the investment puzzle.
This model cannot account for land acquisition costs, which vary
dramatically across Istanbul even within the same district. It also says
nothing about zoning permissions, road access type (a sensor on a
residential back street reads very differently from one on a highway
on-ramp), or environmental constraints near the Bosphorus. Brand
competition dynamics are missing too — a location 1.5 km from an
unnamed independent station carries very different risk than one 1.5 km
from a Shell or BP with loyalty programmes. These factors could easily
flip the ranking if included.

If one additional dataset were available, road classification data from
IBB or OpenStreetMap — specifically whether each sensor sits on a
highway, arterial, or local road — would add the most value. All three
proposed sites are assumed to benefit from high-speed through-traffic,
but that assumption is unverified. Filtering the demand model to
prioritise sensors confirmed to be on arterial or highway segments would
remove low-quality high-count locations (e.g., dense residential blocks
where cars circle slowly but rarely need to refuel) and strengthen
confidence that the selected sites truly serve transit demand rather than
neighbourhood traffic noise.
"""

print(discussion.strip())

print("\n" + "=" * 60)
print("Section 6 complete — Discussion written.")
print("=" * 60)

SECTION 6 — Discussion
The three proposed locations all sit along the TEM motorway corridor
near Ümraniye (approximately 41.09°N, 29.06°E), and that concentration
is not a coincidence. This corridor recorded the highest sustained
vehicle throughput in the entire dataset — the top-3 sites averaged
over 10,000 vehicles per day, compared to a city-wide sensor mean of
roughly 475. What separated them from other high-demand candidates was
the competition gap: Sites 1 and 2 sit more than 1.9 km from any
existing station, and Site 3, despite a slightly closer competitor at
0.52 km, still carries the highest raw DemandScore of all 15 candidates
(0.578). The FinalScore formula — which weights demand at 60% and
competition distance at 40% — rewarded exactly this combination. Several
locations ranked higher on demand alone but fell below the cutoff because
they were already surrounded by rival stations within 300–500 metres,
making them viable for competition but unattractive for a new entrant.



---

### Questions?

**Dr. Eyuphan Koc**  
eyuphan.koc@bogazici.edu.tr